In [ ]:
ADD

Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
Dataset downloaded to: /kaggle/input/plantvillage-dataset


In [ ]:
import os
import sys
import time
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# 1. Hardware setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# 2. Dataset Path (Color classes)
DATA_DIR = Path(path) / "plantvillage dataset" / "color"

# 3. Custom Dataset Loader
class PlantDataset(Dataset):
    def __init__(self, data_dir: Path, transform=None):
        self.transform = transform
        self.samples = []
        valid_dirs = sorted([d for d in data_dir.iterdir() if d.is_dir()])
        self.classes = [d.name for d in valid_dirs]
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        for d in valid_dirs:
            cls_idx = self.class_to_idx[d.name]
            for img in d.iterdir():
                if img.is_file() and img.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                    self.samples.append((str(img), cls_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform:
                img = self.transform(img)
            return img, label

# 4. Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 5. Prepare Train/Val Split (80/20)
full_data = PlantDataset(DATA_DIR)
classes = full_data.classes
total_size = len(full_data)
val_size = int(total_size * 0.2)
train_size = total_size - val_size

train_subset, val_subset = torch.utils.data.random_split(
    full_data, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        with Image.open(path) as img:
            img = img.convert("RGB")
            return self.transform(img), label

train_loader = DataLoader(TransformSubset(train_subset, train_transform), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(TransformSubset(val_subset, val_transform), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"[INFO] 38 Classes | Training images: {train_size} | Validation images: {val_size}")

# 6. Build MobileNetV3-Small
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(in_features, len(classes))
)
model = model.to(device)

# 7. Training Loop (15 Epochs)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

best_acc = 0.0
total_start = time.time()

print("\n[INFO] Starting GPU Training for 15 Epochs...\n")

for epoch in range(1, 16):
    epoch_start = time.time()
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    scheduler.step()
    train_acc = correct / total

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    duration = time.time() - epoch_start
    print(f"Epoch [{epoch:02d}/15] in {duration:.1f}s | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Val Loss: {val_loss/val_total:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc,
            'classes': classes,
            'model_name': 'mobilenet_v3_small'
        }, "disease_model.pth")
        print(f"  ★ Best Model Saved (Accuracy: {val_acc*100:.2f}%)")

print(f"\n[SUCCESS] Completed 15 Epochs in {(time.time()-total_start)/60:.2f} mins! Peak Accuracy: {best_acc*100:.2f}%")


[INFO] Using Device: cuda (Tesla T4)
[INFO] 38 Classes | Training images: 43444 | Validation images: 10861
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 134MB/s]



[INFO] Starting GPU Training for 15 Epochs...

Epoch [01/15] in 356.7s | Train Acc: 93.13% | Val Acc: 96.00% | Val Loss: 0.1198
  ★ Best Model Saved (Accuracy: 96.00%)
Epoch [02/15] in 249.5s | Train Acc: 97.70% | Val Acc: 95.08% | Val Loss: 0.1730
Epoch [03/15] in 244.6s | Train Acc: 98.31% | Val Acc: 95.56% | Val Loss: 0.1472
Epoch [04/15] in 254.2s | Train Acc: 98.63% | Val Acc: 98.21% | Val Loss: 0.0663
  ★ Best Model Saved (Accuracy: 98.21%)
Epoch [05/15] in 246.4s | Train Acc: 98.90% | Val Acc: 98.04% | Val Loss: 0.0658
Epoch [06/15] in 249.4s | Train Acc: 99.11% | Val Acc: 99.27% | Val Loss: 0.0228
  ★ Best Model Saved (Accuracy: 99.27%)
Epoch [07/15] in 249.0s | Train Acc: 99.34% | Val Acc: 96.17% | Val Loss: 0.1283
Epoch [08/15] in 251.6s | Train Acc: 99.46% | Val Acc: 99.13% | Val Loss: 0.0281
Epoch [09/15] in 254.4s | Train Acc: 99.63% | Val Acc: 99.53% | Val Loss: 0.0167
  ★ Best Model Saved (Accuracy: 99.53%)
Epoch [10/15] in 251.0s | Train Acc: 99.78% | Val Acc: 99.62% |

In [ ]:
from google.colab import files
files.download("disease_model.pth")
